In [ ]:
pip install evaluate

In [ ]:
pip install --upgrade transformers

In [ ]:
pip install numpy==1.26.4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset
from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

# Load Jigsaw dataset
jigsaw_dir = Path("/content/drive/My Drive/Jigsaw")
df = pd.read_csv(jigsaw_dir /'train.csv')

# Define binary toxicity label (toxic if any of the categories are 1)
df['toxic_label'] = (df[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']].sum(axis=1) > 0).astype(int)
df = df[['comment_text', 'toxic_label']]
df = df.rename(columns={"comment_text": "text"})
# Split
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Tokenizer & model
checkpoint = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize(example):
    return tokenizer(example['text'], truncation=True, padding='max_length', max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# Set format for Trainer
train_dataset = train_dataset.rename_column("toxic_label", "label")
test_dataset = test_dataset.rename_column("toxic_label", "label")
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Load model
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

# Evaluation metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

# Training configuration
training_args = TrainingArguments(
    output_dir="./roberta-toxic-jigsaw",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir="./logs",
    report_to=[]
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Train
trainer.train()
model_dir = Path("/content/drive/My Drive/DALI/model")
# Save model
model.save_pretrained(model_dir / "roberta-toxic-jigsaw")
tokenizer.save_pretrained(model_dir / "roberta-toxic-jigsaw")

# Evaluate
results = trainer.evaluate()
print("Evaluation:", results)


In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset
from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

model_dir = Path("/content/drive/My Drive/DALI/model")

# Lyrics
lyrics_dir = Path("/content/drive/My Drive/DALI/unzipped_files/lyricsline/all_lyrics_labeled.csv")
lyrics_df = pd.read_csv(lyrics_dir)
lyrics_df = lyrics_df.dropna(subset=['text'])


# Define binary toxicity label (toxic if any of the categories are 1)
# Split
train_df, test_df = train_test_split(
    lyrics_df,
    test_size=0.1,
    random_state=42,
    stratify=lyrics_df["label"]  # Ensures label distribution is preserved
)
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Tokenizer & model
checkpoint = str(model_dir / "roberta-toxic-jigsaw")

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize(example):
    return tokenizer(example['text'], truncation=True, padding='max_length', max_length=128)

train_dataset = train_dataset.map(tokenize)
test_dataset = test_dataset.map(tokenize)

# Set format for Trainer
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Load model
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

# Evaluation metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

# Training configuration
training_args = TrainingArguments(
    output_dir="./roberta-toxic-jigsaw",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir="./logs",
    report_to=[]
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Train
trainer.train()

# Save model
model.save_pretrained(model_dir/"roberta-lyrics")
tokenizer.save_pretrained(model_dir/"roberta-lyrics")

# Evaluate
results = trainer.evaluate()
print("Evaluation:", results)


In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset
from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

model_dir = Path("/content/drive/My Drive/DALI/model")

# Lyrics
lyrics_dir = Path('/content/drive/MyDrive/DALI/clean_explicit_comparison/lyrics_labeled.csv')
lyrics_df = pd.read_csv(lyrics_dir)
lyrics_df = lyrics_df.dropna(subset=['text'])


# Define binary toxicity label (toxic if any of the categories are 1)
# Split
train_df, test_df = train_test_split(
    lyrics_df,
    test_size=0.1,
    random_state=42,
    stratify=lyrics_df["label"]  # Ensures label distribution is preserved
)
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Tokenizer & model
checkpoint = Path("/content/drive/My Drive/DALI/model/roberta-lyrics")

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize(example):
    return tokenizer(example['text'], truncation=True, padding='max_length', max_length=128)

train_dataset = train_dataset.map(tokenize)
test_dataset = test_dataset.map(tokenize)

# Set format for Trainer
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Load model
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

# Evaluation metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

# Training configuration
training_args = TrainingArguments(
    output_dir="./roberta-toxic-jigsaw",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir="./logs",
    report_to=[]
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Train
trainer.train()

# Save model
model.save_pretrained(model_dir/"roberta-lyrics_final")
tokenizer.save_pretrained(model_dir/"roberta-lyrics_final")

# Evaluate
results = trainer.evaluate()
print("Evaluation:", results)
